<a href="https://colab.research.google.com/github/IrumShehryar/ML-NLP-Coursework/blob/main/nlp/04-neural-network/neural-word-embeddings/Project02_ImplementingCBOW_LargeCorpus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Libraries

In [1]:
!pip install wikipedia

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=6cf50d95923e8834957090b812e61090fe40c2916a0791c3f117614d543a3081
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [2]:
from nltk.tokenize import word_tokenize
import numpy as np
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
import wikipedia
import string
import numpy as np
import pandas as pd

# Get the data and tokenize it

In [6]:
import requests
import wikipedia.exceptions

try:
    page_africa = wikipedia.page("Africa")
except requests.exceptions.JSONDecodeError as e:
    print(f"Error decoding JSON from Wikipedia API: {e}")
    print("This might be a temporary issue with the Wikipedia API or network connectivity. Please try again later.")
    page_africa = None # Set to None or handle error as appropriate
except wikipedia.exceptions.WikipediaException as e:
    print(f"An error occurred with the Wikipedia library: {e}")
    page_africa = None
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    page_africa = None


In [7]:
text = ( page_africa.content.translate(str.maketrans('', '', string.punctuation)))

In [8]:
tokenized_sentence = word_tokenize(text.lower())

In [9]:
tokenized_sentence

['the',
 'united',
 'states',
 'of',
 'america',
 'usa',
 'also',
 'known',
 'as',
 'the',
 'united',
 'states',
 'us',
 'or',
 'america',
 'is',
 'a',
 'country',
 'primarily',
 'located',
 'in',
 'north',
 'america',
 'it',
 'is',
 'a',
 'federal',
 'republic',
 'consisting',
 'of',
 '50',
 'states',
 'and',
 'a',
 'federal',
 'capital',
 'district',
 'washington',
 'dc',
 'the',
 '48',
 'contiguous',
 'states',
 'border',
 'canada',
 'to',
 'the',
 'north',
 'and',
 'mexico',
 'to',
 'the',
 'south',
 'with',
 'the',
 'semiexclave',
 'of',
 'alaska',
 'in',
 'the',
 'northwest',
 'and',
 'the',
 'archipelago',
 'of',
 'hawaii',
 'in',
 'the',
 'pacific',
 'ocean',
 'the',
 'united',
 'states',
 'also',
 'asserts',
 'sovereignty',
 'over',
 'five',
 'major',
 'island',
 'territories',
 'and',
 'various',
 'uninhabited',
 'islands',
 'in',
 'oceania',
 'and',
 'the',
 'caribbean',
 'it',
 'is',
 'a',
 'megadiverse',
 'country',
 'with',
 'the',
 'worlds',
 'thirdlargest',
 'land',
 'a

In [10]:
len(tokenized_sentence)

14250

# Generate the training data for CBOW. CBOW predicts the middle word based on the context

In [11]:
training_data = {}
neighbors = 5
for index, word in enumerate(tokenized_sentence):
    if (index < neighbors) or (len(tokenized_sentence)-index < neighbors+1):
        continue
    else:
        start = index-neighbors
        finish = index+neighbors

        neighbor_words = tokenized_sentence[start:finish+1]

        training_data[word] = neighbor_words[:neighbors]+neighbor_words[neighbors+1:]


In [12]:
training_data # key is the word and value is the context

{'usa': ['times',
  'the',
  'washington',
  'post',
  'and',
  'today',
  'about',
  '800',
  'publications',
  'are'],
 'also': ['2026',
  'fifa',
  'world',
  'cup',
  'see',
  'lists',
  'of',
  'us',
  'state',
  'topics'],
 'known': ['elsewhere',
  'the',
  'academy',
  'awards',
  'popularly',
  'as',
  'the',
  'oscars',
  'have',
  'been'],
 'as': ['sports',
  'with',
  'significant',
  'exceptions',
  'such',
  'minor',
  'league',
  'baseball',
  'this',
  'differs'],
 'the': ['the',
  'interior',
  'wikimedia',
  'atlas',
  'of',
  'united',
  'states',
  'geographic',
  'data',
  'related'],
 'united': ['states',
  'geographic',
  'data',
  'related',
  'to',
  'states',
  'at',
  'openstreetmap',
  'measure',
  'of'],
 'states': ['geographic',
  'data',
  'related',
  'to',
  'united',
  'at',
  'openstreetmap',
  'measure',
  'of',
  'america'],
 'us': ['–',
  'official',
  'maps',
  'from',
  'the',
  'department',
  'of',
  'the',
  'interior',
  'wikimedia'],
 'or': [

# Create Vocab Dictionary

In [13]:
vocab = list(set(tokenized_sentence))

In [15]:
vocab.sort()
vocab

['03',
 '06',
 '07',
 '1',
 '10',
 '100',
 '1000',
 '10000',
 '100000',
 '100th',
 '102',
 '104',
 '105',
 '107',
 '11',
 '11000',
 '1100000',
 '115',
 '117',
 '1179',
 '118',
 '12',
 '121',
 '123',
 '1277',
 '12th',
 '13',
 '13200',
 '135',
 '138',
 '139',
 '13th',
 '14',
 '14000',
 '142',
 '14224',
 '144',
 '1454–1512',
 '1492',
 '15',
 '15000',
 '1500s',
 '1507',
 '1513',
 '1513–1765',
 '152',
 '15460',
 '1562',
 '1565',
 '1566',
 '1578',
 '159',
 '16',
 '1607',
 '1620',
 '1626',
 '1628',
 '1638',
 '16700',
 '169',
 '16th',
 '17',
 '17000',
 '1701',
 '171',
 '1718',
 '1764',
 '1765–1783',
 '1765–1800',
 '1770',
 '1770s',
 '1774',
 '1775',
 '1775–1783',
 '1776',
 '1777',
 '1781',
 '1783',
 '1787',
 '1789',
 '1791',
 '1793',
 '17th',
 '18',
 '180',
 '1800',
 '18000',
 '1800–1865',
 '1803',
 '1812',
 '1819',
 '1820',
 '1824123',
 '1830',
 '1830s',
 '1830–1850',
 '1845',
 '1846',
 '1846–1848',
 '1847',
 '1848',
 '1848–1849',
 '185',
 '1850',
 '1850s',
 '1851',
 '1854',
 '1857',
 '1861',

In [17]:
vocab_dict = {}

for index, word in enumerate(vocab):
    vocab_dict[word] = index
vocab_dict

{'03': 0,
 '06': 1,
 '07': 2,
 '1': 3,
 '10': 4,
 '100': 5,
 '1000': 6,
 '10000': 7,
 '100000': 8,
 '100th': 9,
 '102': 10,
 '104': 11,
 '105': 12,
 '107': 13,
 '11': 14,
 '11000': 15,
 '1100000': 16,
 '115': 17,
 '117': 18,
 '1179': 19,
 '118': 20,
 '12': 21,
 '121': 22,
 '123': 23,
 '1277': 24,
 '12th': 25,
 '13': 26,
 '13200': 27,
 '135': 28,
 '138': 29,
 '139': 30,
 '13th': 31,
 '14': 32,
 '14000': 33,
 '142': 34,
 '14224': 35,
 '144': 36,
 '1454–1512': 37,
 '1492': 38,
 '15': 39,
 '15000': 40,
 '1500s': 41,
 '1507': 42,
 '1513': 43,
 '1513–1765': 44,
 '152': 45,
 '15460': 46,
 '1562': 47,
 '1565': 48,
 '1566': 49,
 '1578': 50,
 '159': 51,
 '16': 52,
 '1607': 53,
 '1620': 54,
 '1626': 55,
 '1628': 56,
 '1638': 57,
 '16700': 58,
 '169': 59,
 '16th': 60,
 '17': 61,
 '17000': 62,
 '1701': 63,
 '171': 64,
 '1718': 65,
 '1764': 66,
 '1765–1783': 67,
 '1765–1800': 68,
 '1770': 69,
 '1770s': 70,
 '1774': 71,
 '1775': 72,
 '1775–1783': 73,
 '1776': 74,
 '1777': 75,
 '1781': 76,
 '1783': 77

# Create one hot vectors for each word

In [18]:
vocab_arrays = {}

for word, column_index in vocab_dict.items():
    word_array = np.zeros((1, len(vocab_dict)))
    word_array[0, column_index] = 1
    vocab_arrays[word] = word_array

# Generate Features and labels to train Neural Network

In [19]:
word_size = len(vocab_dict)
training_size = len(training_data.items())
print(word_size)
print(" ")
print(training_size)

3612
 
3612


### Intialize features and labels

In [21]:
# Initialize the context vector
X = np.zeros([training_size, word_size])

# Initialize the word vectorto be predicted
y = np.zeros([training_size, word_size])

X
y

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

# Fill in X and y and take average of context vector

In [22]:
for index, train in enumerate(training_data.items()):

    word = train[0]
    y[index,:] = vocab_arrays[word]

    temp_array = np.zeros([1, word_size])

    for neighbour in train[1]:
        temp_array = temp_array+vocab_arrays[neighbour]

    X[index,:] = temp_array/(neighbors*2)

In [23]:
X

array([[0. , 0. , 0. , ..., 0. , 0. , 0. ],
       [0. , 0. , 0. , ..., 0. , 0. , 0. ],
       [0. , 0. , 0. , ..., 0. , 0. , 0. ],
       ...,
       [0. , 0. , 0. , ..., 0. , 0. , 0.1],
       [0. , 0. , 0. , ..., 0. , 0. , 0.1],
       [0. , 0. , 0. , ..., 0. , 0. , 0. ]])

# Create and compile Neural Network

In [24]:
from keras.models import Sequential
from keras.layers import Dense

In [25]:
# Building model architecture
model = Sequential()
# Adding the hidden layer
model.add(Dense(40, input_dim=word_size, activation='relu')) # The 40 here is our choice. it is hidden layer.
# Adding the output layer
model.add(Dense(word_size, input_dim=40, activation='softmax'))

# Compiling the network
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


# Train the NN

In [26]:
# Fitting our model, let's train for 1000
# epochs
model.fit(X, y, epochs=100)

Epoch 1/100
113/113 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.0000e+00 - loss: 8.2125
Epoch 2/100
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.0000e+00 - loss: 8.1951
Epoch 3/100
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.0053 - loss: 8.1753
Epoch 4/100
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.0144 - loss: 8.0873
Epoch 5/100
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.0266 - loss: 7.9144
Epoch 6/100
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.0476 - loss: 7.6771
Epoch 7/100
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.0772 - loss: 7.3932
Epoch 8/100
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.1409 - loss: 7.0762
Epoch 9/100
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.2262 - loss: 6.7329
Epoch 10/100
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.3120 - loss: 6.3726
Epoch 11/100
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.3995 - loss: 6.0019
Epoch 12/100
113/113 ━━━━━━━━━━━

# Get the predictions

In [27]:
# Storing the prediction
prediction = model.predict(X)

113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [28]:
prediction.shape

(3612, 3612)

### Let's check what word our model predicts as the center word given the surrounding words:

In [29]:
for index, sentence in enumerate(training_data.items()):
  if index == 4:
    break
    print('For sentence: '+' '.join(sentence[1][0:2])+' '+' '.join(sentence[1][2:]))
    predicted_word = prediction[index]

    # Store keys and values from our dictionary
    for word, array_val in vocab_arrays.items():
        if np.array_equal(array_val[0], predicted_word):
            print('The center word predicted is: '+word)

# Check the embeddings which are our word vectors

In [30]:
embeddings = [layer.get_weights() for layer in model.layers]

In [31]:
embeddings

[[array([[ 0.01853503, -0.6221058 ,  0.48435518, ..., -0.46019047,
          -0.19300951,  0.71466064],
         [ 0.89990324,  0.71142673,  0.13386665, ...,  0.6738511 ,
           0.17156816, -1.2189815 ],
         [ 0.02404979,  0.8614017 ,  0.42688778, ..., -0.01257207,
          -0.33957127,  0.78120357],
         ...,
         [-0.43528727,  0.8154914 ,  1.0870962 , ..., -0.645903  ,
          -0.7887005 ,  1.424786  ],
         [-0.1659038 , -0.43451828,  0.5975313 , ..., -0.87124205,
           0.336227  ,  0.4277017 ],
         [ 0.38809118,  1.4797316 ,  1.5121596 , ...,  0.5593789 ,
           1.6844273 , -1.823111  ]], dtype=float32),
  array([0.9688271 , 1.2102108 , 1.3997823 , 0.8153542 , 0.71425116,
         0.88752264, 0.7068783 , 1.2220433 , 0.77057064, 1.1578454 ,
         0.96829516, 1.0048379 , 1.0133778 , 0.7491117 , 1.0136689 ,
         0.9863499 , 0.8420189 , 1.7209334 , 0.97411346, 1.0141354 ,
         1.2780815 , 0.8842438 , 1.6222292 , 0.875569  , 1.2097552 ,


In [32]:
embeddings_layer_1 = embeddings[0][0]
embeddings_layer_2 = embeddings[1][0]

### Word vector or word embeddings is the average of both embeddings

In [33]:
word_embeddings = (embeddings_layer_2.T+embeddings_layer_1)/2

In [34]:
word_embeddings = pd.DataFrame(word_embeddings, index=vocab)

In [35]:
word_embeddings

,0,1,2,3,4,5,6,7,8,9,...,30,31,32,33,34,35,36,37,38,39
03,-0.645326,-0.362615,0.165195,-0.731243,-0.281071,-1.436679,0.183732,-0.632993,-2.708687,0.210163,...,0.803137,0.539852,-0.063159,-0.244411,-0.062554,0.268261,-0.457955,-0.905343,-0.127168,0.408356
06,0.362319,0.200434,-0.138213,0.208476,-1.616126,0.266938,-0.599879,0.341775,0.421661,-0.711984,...,-0.384729,-0.152485,-0.543410,0.416623,0.519514,0.092318,0.154837,0.233427,-0.077210,-0.987623
07,0.044636,0.288290,-0.017260,0.170324,-0.000820,0.338250,-2.413638,0.482998,-0.050994,-2.761599,...,0.253308,-0.079968,0.039924,-0.820301,0.396320,0.425730,-1.322319,-0.352590,-0.130806,0.288479
1,-1.559667,0.289520,0.317322,0.558791,-0.260499,-0.505186,-0.344271,-1.279764,-0.493864,0.871386,...,0.463412,-1.667096,-0.470045,-0.628108,0.464478,-0.091457,0.040133,0.005650,0.421079,0.657678
10,-0.429880,0.093028,-0.219513,-1.220265,0.318222,0.511034,-0.845432,-0.342809,0.004794,1.062268,...,-0.620746,-0.703378,1.078683,0.930296,1.201820,-1.167370,0.129119,0.630961,0.277525,-0.267546
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
youtube,-0.470426,0.157684,-0.112222,-1.118412,0.321848,-1.282652,-0.030830,-0.674613,0.083607,0.566877,...,-1.116956,0.327693,0.075753,0.641926,-1.506235,0.442450,0.352520,0.529155,-0.337916,-0.767243
zealand,0.196899,-0.120173,-0.577787,0.358463,-3.166583,0.281399,0.308751,0.287462,-0.466558,-0.241647,...,0.404518,0.180152,0.328571,0.335595,-0.995156,0.011730,-0.631934,-0.356782,0.198185,0.249101
zee,-1.614949,0.472615,0.579307,-0.367426,-1.547199,0.883383,-1.137976,-0.058864,0.150144,0.812074,...,0.630754,-2.275297,-0.095042,-0.784235,-0.553150,-1.079659,0.505117,-0.504017,-0.334186,0.733602
zones,-0.097375,-0.312302,0.313916,0.551822,0.293260,0.346501,0.534009,-0.577580,-1.019625,0.310410,...,0.243894,0.467945,0.545210,-1.452067,-1.000441,-1.344446,-0.759101,-0.828079,-0.030076,-0.060361


# Finding the Cosine Similarity between word embeddings

In [36]:
word_embeddings_sim = pd.DataFrame(
    cosine_similarity(word_embeddings),
    columns=vocab,
    index=vocab
)

In [37]:
def get_most_similar_words(word):
    return (
        word_embeddings_sim[word].sort_values(ascending=False).head(10),
    )

In [38]:
get_most_similar_words('great')

(great          1.000000
 write          0.736361
 attempt        0.695835
 plains         0.627968
 colonists      0.593025
 disarm         0.560817
 novel          0.559918
 amendment      0.558995
 eastern        0.550814
 subtropical    0.544157
 Name: great, dtype: float32,)

In [39]:
get_most_similar_words('mexico')

(mexico       1.000000
 canada       0.791158
 currently    0.646999
 2026         0.646802
 cohosting    0.559169
 see          0.511388
 1628         0.505709
 1718         0.477887
 fifa         0.477574
 now          0.469028
 Name: mexico, dtype: float32,)